#### Training Track - Quantum Device Design Workshop, June 15–18, 
---

**Author**: Jens Koch
# Circuit Analysis and Simulation using "scqubits"


*scqubits is an open-source Python package that helps with the modeling of superconducting circuits:*
- common superconducting qubits: `Transmon`, `TunableTransmon`, `Fluxonium` and more
- new, user-defined circuits
- coupled systems of superconducting circuits

This brief tutorial focuses on getting you started with the modeling of individual qubits.
Depending on your knowledge, you may or may not finish the activities suggested throughout the notebook. Feel free to fast-forward through content you are well-familiar with.


## 1. Exploring commmon superconducting qubits with the GUI
scqubits exposes a subset of its single-qubit modeling capabilities in a beginner-friendly graphical user interface (GUI).


Launch it:

In [ ]:
# One-time environment shim (scqubits 4.3.1): the Oscillator widget loads
# 'oscillator.jpg', but the package ships 'Oscillator.jpg' — a case mismatch that
# breaks scq.Oscillator.create() on Linux. Create the lowercase alias if missing.
import os, shutil, scqubits
_img = os.path.join(os.path.dirname(scqubits.__file__), 'core', 'qubit_img')
_alias = os.path.join(_img, 'oscillator.jpg')
if not os.path.exists(_alias):
    try:
        shutil.copyfile(os.path.join(_img, 'Oscillator.jpg'), _alias)
    except OSError as _e:
        print('Note: could not create oscillator.jpg alias:', _e)


In [ ]:
import scqubits as scq
scq.GUI()

<div class="alert alert-block alert-info">
<b>Activities 1: Familiarize yourself with the GUI</b>

Some questions/steps that may guide your exploration: 
- Which qubits are included in the GUI?
  
- Switch between qubits, pick one you are less familiar with, and note that the GUI has a tab providing basic information about the selected qubit.

- What are the different kind of plots the GUI can generate? How might they be useful in device design, device characterization, and qubit operations?

- Play with sliders and note how changes in parameters affect properties of qubits. Examples to consider: for the `Transmon`, note how a reduced $E_J/E_C$ increases the "charge dispersion" (i.e., the dependence of eigenenergies on the offset charge $n_g$). For `Fluxonium`, observe how decreasing the charging energy $E_C$ tends to localize wavefunctions (as it increases the "effective mass").
</div>



## 2. Exploring spectral properties of the `Transmon` qubit

The GUI is a nice starting point. For most of your quantitative work, however, you may rather want to switch to the programmatic way of using scqubits. 

Here is how to create a `Transmon` qubit instance:

In [ ]:
tmon = scq.Transmon(EJ=30.0, EC=1.2, ng=0.0, ncut=31)

From within a jupyter, there is also a convenience option that updates qubit parameters each time values are changed in the entry fields:

In [ ]:
tmon = scq.Transmon.create()


<div class="alert alert-block alert-info">
<b>Activities 2: Explore transmon spectra</b>

- Obtain the eigenenergies for the ground state and lowest-lying excited states via `tmon.eigenvals()`.
  
- Compare the energy splitting $E_{01}$ between ground and first excited state. How well does it match $\sqrt{8E_JE_C}$? Is  $\sqrt{8E_JE_C} -E_C$ a better approximation?

- Given the parameters you selected, what level of thermal excitation (occupation probability for level 1) do you expect at a temperature of 20mK (100mK)?

- How anharmonic is your transmon? Compute the so-called "absolute anharmonicity" $\alpha=E_{12} - E_{01}$. How close is this to $-E_C$?

- Suppose you have a target qubit frequency $E_{01}$ and a desired anharmonicity $\alpha$. How would you go about choosing $E_J$ and $E_C$? `Transmon` has a convenience method for this, check out __[.find_EJ_EC(E01, alpha)](https://scqubits.readthedocs.io/en/latest/api-doc/_autosummary/scqubits.core.transmon.Transmon.html#scqubits.core.transmon.Transmon.find_EJ_EC)__
</div>

## 3. Towards simulation of driven and coupled systems: matrix elements

Can you use a drive to manipulate your qubit's quantum state and perform, say, an $X$ gate that maps $|0\rangle \mapsto|1\rangle$ and $|1\rangle \mapsto|0\rangle$? 

Or, if you couple a qubit to a resonator mode, what are the dressed eigenstates and how much/what kind of hybridization do you expect? 

Much of this depends on matrix elements: the drive and the resonator mode couple to the qubit via some qubit operator $\hat{M}$. For example, the qubit charge number operator $\hat{M}=\hat{n}$ is commonly encountered here when coupling to the qubit via a coupling capacitor. (For the coupling of atoms to the electromagnetic field, the operator might be the electric dipole moment.) Depending on the nature of the qubit's eigenstates, there may or may not be selection rules that can prevent certain transitions.

In short, we are interested in matrix elements like $\langle 0 | \hat{n} | 1\rangle$.

In scqubits, every qubit features its own relevant operators. For example, here is the transmon charge (number) operator:

In [ ]:
tmon.n_operator()

Here, we get a glimpse at the inner life of scqubits: the $\hat{n}$ operator for the transmon is represented in charge basis where $\hat{n}$ is diagonal. For the most part, we do not have to be concerned with this internal representation (except for the matter of convergence with respect to the cutoff `ncut` which reduces the infinite-dimensional Hilbert space to a truncated, finite-dimensional one).

The matrix elements we are interested in are with respect to qubit eigenstates, and scqubits provides a couple of convenient methods for computing and visualizing those.

Try: `tmon.matrixelement_table('n_operator')` and `tmon.plot_matrixelements('n_operator')`

<div class="alert alert-block alert-info">
<b>Activities 3: Inspecting matrix elements</b>
    
- Based on the magnitude of charge matrix elements $|\langle j | \hat{n} | j'\rangle|$ (with $j,j'$ labeling the transmon eigenstates), how difficult is it to drive the transmon from $|0\rangle$ to $|1\rangle$, $|2\rangle$ and $|3\rangle$?
    
- Discuss selection rules. Which selection rule(s) is/are specific to the transmon regime, which ones are universal (and therefore hold in the charge regime $E_J<E_C$ as well?)
</div>

## 4. Coupling a transmon to an oscillator: the `HilbertSpace` class

In our final section of this short tutorial, we will couple a transmon to a harmonic-oscillator mode. scqubits enables us to collect multiple quantum systems inside a `HilbertSpace` object. 

In case of capacitive coupling, the full system Hamiltonian takes on the following form:
$$
\hat{H} = \hat{H}_\text{tmon} + \hat{H}_\text{osc} + g\, \hat{n}(\hat{a}+\hat{a}^\dagger).
$$

We first implement a flux-tunable transmon along with an oscillator:

In [ ]:
tmon = scq.TunableTransmon.create()

In [ ]:
osc = scq.Oscillator.create()

From inside a jupyter notebook, there is also a convenience function that allows us to generate the `HilbertSpace` object including the desired interaction term.

In [ ]:
hspace = scq.HilbertSpace.create()

<div class="alert alert-block alert-info">
<b>Activities 4: Coupled systems, `HilbertSpace` class, dressed eigenenergies</b>
    
- Couple the transmon and oscillator mode as suggested, with a coupling strength of $g/h = 150$MHz. 

    
- Think carefully about units: we are essentially working in units where Planck's constant $h$ is set to 1. With this, we may specify energies like $E_J$ in GHz. 
Note: when simulating time dynamics, it is tempting to set $\hbar=1$. Obviously, you cannot set both to 1 simultaneously and factors of $2\pi$ must appear in one place or the other. 
    
- What should be the dimensionality of your joint Hilbert space? What does `hspace.dimension` give you instead and why?

- Obtain the dressed eigenenergies via `hspace.eigenvals()`. How do they relate to the eigenenergies of the decoupled oscillator and transmon systems? Would you say the coupling is dispersive? Can you extract quantities like Lamb shift, and ac Stark shift (see the bottom of __[this documentation page](https://scqubits.readthedocs.io/en/latest/guide/parametersweep/ipynb/paramsweep2-dispersive.html)__ for inspiration.

- With some more time, the next natural extension is to consider running a `ParameterSweep` for the coupled system...

</div>